# Rodada 4 — curvatura predefinida e margens agrícolas

Grau dois fixado antes da estimação, sem seleção de nós. Margens extensiva e intensiva condicionada não identificam decisão estrutural de conservação.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0,str(Path.cwd().parent/'testes2'))
from metodos import *
from IPython.display import display
e=Experimento(Path.cwd());d=e.d

## 1. Preço quadrático × aptidão — dois destinos

In [2]:
for c in ['soja','milho']:d['quadrado_'+c]=d['z_p_'+c]**2*d['z_a_'+c]
x=['expo_soja','expo_milho','quadrado_soja','quadrado_milho']
for u in ['agricultura','pastagem']:
    ident='N01_'+u;r=e.linear(ident,'pp_'+u,x,d,familia='naolinear_zeros',joint=['quadrado_soja','quadrado_milho'])
    e.mag=[a for a in e.mag if a['id']!=ident]
    for cultura in ['soja','milho']:
        for z in [-1,0,1]:
            assert d['z_p_'+cultura].min()<=z<=d['z_p_'+cultura].max()
            e.contraste(ident,f'Derivada diferencial {cultura}; preço z={z}',x,r['b'],r['cov']['DK2'],{'expo_'+cultura:1,'quadrado_'+cultura:2*z},r['s'],'pp por DP preço×DP aptidão','derivadas')
display(pd.DataFrame(e.mag)[['id','contraste','estimativa','ic95_inf','ic95_sup']])

,id,contraste,estimativa,ic95_inf,ic95_sup
0,N01_agricultura,Derivada diferencial soja; preço z=-1,0.085625,0.026325,0.144925
1,N01_agricultura,Derivada diferencial soja; preço z=0,0.031109,0.005631,0.056586
2,N01_agricultura,Derivada diferencial soja; preço z=1,-0.023407,-0.051791,0.004976
3,N01_agricultura,Derivada diferencial milho; preço z=-1,0.028319,-0.002061,0.058700
4,N01_agricultura,Derivada diferencial milho; preço z=0,0.015740,0.000017,0.031463
5,N01_agricultura,Derivada diferencial milho; preço z=1,0.003161,-0.004459,0.010781
6,N01_pastagem,Derivada diferencial soja; preço z=-1,0.123601,-0.014714,0.261916
7,N01_pastagem,Derivada diferencial soja; preço z=0,0.103203,0.040885,0.165522
8,N01_pastagem,Derivada diferencial soja; preço z=1,0.082806,-0.052399,0.218010
9,N01_pastagem,Derivada diferencial milho; preço z=-1,0.098585,-0.059715,0.256885


## 2. Ocorrência de conversão agrícola

VD em 0/100; coeficientes são pp de probabilidade no modelo linear, sem garantia de predições limitadas.

In [3]:
d['ocorrencia_pp']=(100*d.ha_agricultura.gt(0)).where(d.ha_agricultura.notna())
e.linear('Z01_agricultura','ocorrencia_pp',['expo_soja','expo_milho'],d,familia='naolinear_zeros')

{'meta': {'id': 'Z01_agricultura',
  'vd': 'ocorrencia_pp',
  'vi': 'expo_soja + expo_milho',
  'estimador': 'TWFE linear',
  'fe': 'municipio + ano',
  'N': 2772,
  'N_grade': 3384,
  'perdas_grade': 612,
  'municipios': 137,
  'anos': 24,
  'ano_min': np.int64(2001),
  'ano_max': np.int64(2024),
  'K_total': np.int64(162),
  'rank_regressores': np.int64(2),
  'condicao': 1.9566020813003673,
  'iteracoes': 12,
  'convergiu': True,
  'ajuste': np.float64(1.479135399029552e-05),
  'ajuste_tipo': 'R2 within',
  'familia': 'naolinear_zeros',
  'singletons': 3,
  'informativos': 134},
 's':            id         municipio   ano     bioma  ha_agricultura  ha_pastagem  \
 0     5100102          Acorizal  2001   Cerrado        5.093917  1030.491067   
 1     5100102          Acorizal  2002   Cerrado        1.468305   693.661899   
 2     5100102          Acorizal  2003   Cerrado        0.172670   791.607567   
 3     5100102          Acorizal  2004   Cerrado        0.000000   778.872226   
 4

## 3. Taxa agrícola estritamente positiva

Log apenas em observações positivas. A amostra é selecionada pela VD, logo a associação condicionada não pode ser generalizada aos zeros.

In [4]:
d['log_taxa_positiva']=np.log(d.taxa_agricultura.where(d.taxa_agricultura.gt(0)))
r=e.linear('Z02_agricultura','log_taxa_positiva',['expo_soja','expo_milho'],d,familia='naolinear_zeros')
e.mag=[a for a in e.mag if a['id']!='Z02_agricultura']
for cultura in ['soja','milho']:
    pv=d.drop_duplicates('ano')['p_'+cultura];aq=d.drop_duplicates('id')['z_a_'+cultura].quantile([.25,.75]);mult=.1*pv.mean()/pv.std(ddof=0)*(aq.loc[.75]-aq.loc[.25])
    e.contraste('Z02_agricultura','P75−P25; +10% preço '+cultura,r['x'],r['b'],r['cov']['DK2'],{'expo_'+cultura:mult},r['s'],'log taxa condicional positiva','magnitudes_positivas')
display(pd.DataFrame(e.coef).tail(4)[['id','termo','coef','ic95_inf','ic95_sup','p','N']])
e.salvar('naolinear')

,id,termo,coef,ic95_inf,ic95_sup,p,N
8,Z01_agricultura,expo_soja,0.040629,-1.114995,1.196253,0.942650,2772
9,Z01_agricultura,expo_milho,-0.119429,-1.281230,1.042371,0.833472,2772
10,Z02_agricultura,expo_soja,-0.018128,-0.211442,0.175186,0.847889,2507
11,Z02_agricultura,expo_milho,0.069448,-0.045859,0.184756,0.225335,2507
